# Silver Cleaning (A): sifariş axını

**Məsuliyyətli:** Manaf

| Bronze | Silver |
|---|---|
| `bronze.olist_customers_dataset` | `silver.customers` |
| `bronze.olist_orders_dataset` | `silver.orders` |
| `bronze.olist_order_items_dataset` | `silver.order_items` |
| `bronze.olist_order_payments_dataset` | `silver.order_payments` |
| `bronze.api_holidays` | `silver.holidays` |




In [7]:
import sys
sys.path.append("/workspaces/olist-ecommerce-analytics/notebooks")

import pandas as pd
from db import get_engine

In [8]:
import time
from datetime import date, timedelta
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError
from sqlalchemy.engine import URL
import sys
sys.path.append("/workspaces/olist-ecommerce-analytics/notebooks")
from db import get_engine
import os
import json


### customers yoxlama 


In [9]:
df = pd.read_sql("SELECT * FROM bronze.olist_customers_dataset", get_engine())
df.shape # Sətir və sütun sayını yoxlayırıq

(99441, 5)

In [10]:
df.dtypes # Sütunların məlumat tiplərinə baxırıq

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix    object
customer_city               object
customer_state              object
dtype: object

In [11]:
(df == "").sum()# Boş mətnlərin sayını yoxlayırıq

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [12]:
df.isna().sum()# NULL dəyərlərin sayını yoxlayırıq

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [13]:
df.duplicated().sum()# Tam təkrarlanan sətirləri yoxlayırıq

np.int64(0)

In [14]:
df["customer_id"].duplicated().sum()# Təkrarlanan customer_id dəyərlərini yoxlayırıq

np.int64(0)

In [15]:
df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [16]:

# Ştat kodlarının uzunluğunu yoxlayırıq
print(df["customer_state"].str.len().value_counts())

# ZIP kodlarının uzunluğunu yoxlayırıq
print(df["customer_zip_code_prefix"].astype("string").str.len().value_counts())

# Şəhər adlarının kənar boşluqlarını yoxlayırıq
print((df["customer_city"] != df["customer_city"].str.strip()).sum())


customer_state
2    99441
Name: count, dtype: int64
customer_zip_code_prefix
5    99441
Name: count, dtype: Int64
0


In [17]:

from sqlalchemy.types import VARCHAR, CHAR

customers = pd.read_sql(
    "SELECT * FROM bronze.olist_customers_dataset",
    get_engine())

# Məlumatları Silver cədvəlinə yazırıq
customers.to_sql("customers",get_engine(),schema="silver",if_exists="replace",index=False,
    dtype={"customer_id": VARCHAR(32),
        "customer_unique_id": VARCHAR(32),
        "customer_zip_code_prefix": CHAR(5),
        "customer_city": VARCHAR(100),
        "customer_state": CHAR(2)})


138

In [18]:
# Silver cədvəlindəki sətir sayını göstəririk
print(pd.read_sql(
    "SELECT COUNT(*) AS n FROM silver.customers",
    get_engine()
).iloc[0, 0])

99441


### orders

In [36]:
orders = pd.read_sql("SELECT * FROM bronze.olist_orders_dataset", get_engine())
orders.shape

(99441, 8)

In [37]:
# Boş mətnlərin sayını yoxlayırıq
(orders == "").sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [38]:
# Hər sütundakı NULL dəyərləri yoxlayırıq
orders.isna().sum()

order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

In [39]:
# Tam təkrarlanan sətirlərin sayını yoxlayırıq
orders.duplicated().sum()

np.int64(0)

In [40]:
# Təkrarlanan sifariş ID-lərini yoxlayırıq
orders["order_id"].duplicated().sum()

np.int64(0)

In [41]:
# Hər sifariş statusunun sayını yoxlayırıq
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [42]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [44]:
# Sifarişin əsas mərhələlərinə aid tarix sütunlarını seçirik
date_cols = [
    "order_purchase_timestamp",          # Sifarişin verildiyi vaxt
    "order_approved_at",                 # Sifarişin təsdiqləndiyi vaxt
    "order_delivered_carrier_date",      # Kuryerə təhvil verildiyi vaxt
    "order_delivered_customer_date",     # Müştəriyə çatdırıldığı vaxt
    "order_estimated_delivery_date"      # Təxmini çatdırılma tarixi
]

Tarix sütunlarının yoxlanılması zamanı düzgün formata uyğun olmayan dəyər aşkarlanmadı. `order_purchase_timestamp` və `order_estimated_delivery_date` sütunlarında boş dəyər yoxdur. Digər üç sütunda isə boş dəyərlər mövcuddur. Bu boşluqlar sifarişin hələ təsdiqlənməməsi və ya çatdırılma mərhələsinin tamamlanmaması ilə əlaqəli ola bilər. Buna görə həmin dəyərləri avtomatik silmək əvəzinə, sifariş statusları ilə birlikdə yoxlamaq lazımdır.

In [ ]:

# Hər tarix sütununda boş və səhv formatlı tarixləri yoxlayırıq
for col in date_cols:
    empty = orders[col].isna().sum() + (orders[col] == "").sum()

    dates = pd.to_datetime(
        orders[col].replace("", pd.NA),
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce")

    bad = dates.isna().sum() - empty

    print(f"{col}: boş = {empty}, səhv format = {bad}")



order_purchase_timestamp: boş = 0, səhv format = 0
order_approved_at: boş = 160, səhv format = 0
order_delivered_carrier_date: boş = 1783, səhv format = 0
order_delivered_customer_date: boş = 2965, səhv format = 0
order_estimated_delivery_date: boş = 0, səhv format = 0


In [49]:
# Tarixlərin boş olub-olmadığını yoxlayırıq
check = orders.assign(
    approved_empty=orders["order_approved_at"].isna() | (orders["order_approved_at"] == ""),
    carrier_empty=orders["order_delivered_carrier_date"].isna() | (orders["order_delivered_carrier_date"] == ""),
    customer_empty=orders["order_delivered_customer_date"].isna() | (orders["order_delivered_customer_date"] == "")
)

# Statuslara görə boş tarixləri sayırıq
check.groupby("order_status")[
    ["approved_empty", "carrier_empty", "customer_empty"]
].sum()

,approved_empty,carrier_empty,customer_empty
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [52]:
orders[date_cols].dtypes

order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [53]:
# Tarix sütunlarını datetime tipinə çeviririk
for col in date_cols:
    orders[col] = pd.to_datetime(
        orders[col].replace("", pd.NA),
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )

In [54]:
orders[date_cols].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [61]:

from sqlalchemy.types import VARCHAR
from sqlalchemy.dialects.mssql import DATETIME2

orders = pd.read_sql(
    "SELECT * FROM bronze.olist_orders_dataset",
    get_engine())

# Tarix sütunlarını seçirik
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"]

# Tarix sütunlarını datetime tipinə çeviririk
for col in date_cols:
    orders[col] = pd.to_datetime(
        orders[col].replace("", pd.NA),
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce")

# Məlumatları Silver cədvəlinə yazırıq
orders.to_sql("orders",get_engine(),schema="silver",if_exists="replace",index=False,
    dtype={
        "order_id": VARCHAR(32),
        "customer_id": VARCHAR(32),
        "order_status": VARCHAR(20),
        **{col: DATETIME2(precision=0) for col in date_cols}})


143

In [62]:
# Silver cədvəlindəki sifarişlərin sayını yoxlayırıq
pd.read_sql(
    "SELECT COUNT(*) AS n FROM silver.orders",get_engine())

,n
0,99441


### order_items

In [64]:
items = pd.read_sql("SELECT * FROM bronze.olist_order_items_dataset",get_engine())
items.shape

(112650, 7)

In [66]:
(items == "").sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [67]:
# dublikatların yoxlanılması
items.duplicated().sum()

np.int64(0)

In [68]:
# Sifariş ID-si və məhsul sıra nömrəsinin təkrarını yoxlayırıq
items.duplicated(subset=["order_id", "order_item_id"]).sum()

np.int64(0)

In [69]:
# Sütunların hazırkı məlumat tiplərini yoxlayırıq
items[["price", "freight_value"]].dtypes

price            object
freight_value    object
dtype: object

In [73]:
# Qiymət və çatdırılma haqqını rəqəm tipinə çeviririk
items["price"] = pd.to_numeric(items["price"], errors="coerce")
items["freight_value"] = pd.to_numeric(items["freight_value"], errors="coerce")

In [74]:
items[["price", "freight_value"]].dtypes

price            float64
freight_value    float64
dtype: object

In [75]:
# Rəqəm sütunlarında boş dəyərlərin sayını yoxlayırıq
items[["price", "freight_value"]].isna().sum()

price            0
freight_value    0
dtype: int64

In [24]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [76]:
# Qiymət və çatdırılma haqqının minimum və maksimumunu yoxlayırıq
price.min(), price.max(), freight.min(), freight.max()

(np.float64(0.85), np.float64(6735.0), np.float64(0.0), np.float64(409.68))

In [80]:
# Çatdırılma üçün son tarixi datetime tipinə çeviririk
items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"].replace("", pd.NA),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce")

In [82]:
items["shipping_limit_date"].dtype

dtype('<M8[ns]')

In [83]:
# Boş və tarixə çevrilməyən dəyərləri yoxlayırıq
items["shipping_limit_date"].isna().sum()

np.int64(0)

In [84]:
from sqlalchemy import types as sqltypes
from sqlalchemy.dialects.mssql import DATETIME2

items = pd.read_sql("SELECT * FROM bronze.olist_order_items_dataset",get_engine())

# Sütunları uyğun məlumat tiplərinə çeviririk
items["order_item_id"] = pd.to_numeric(items["order_item_id"])
items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"].replace("", pd.NA),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce")

items["price"] = pd.to_numeric(items["price"], errors="coerce")
items["freight_value"] = pd.to_numeric(items["freight_value"], errors="coerce")

# Təmizlənmiş məlumatları Silver cədvəlinə yazırıq
items.to_sql("order_items",get_engine(),schema="silver",if_exists="replace",index=False,
    dtype={
        "order_id": sqltypes.VARCHAR(32),
        "order_item_id": sqltypes.SMALLINT(),
        "product_id": sqltypes.VARCHAR(32),
        "seller_id": sqltypes.VARCHAR(32),
        "shipping_limit_date": DATETIME2(precision=0),
        "price": sqltypes.DECIMAL(10, 2),
        "freight_value": sqltypes.DECIMAL(10, 2)})

226

In [86]:
items_silver.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [90]:
items_silver = pd.read_sql(
    "SELECT * FROM silver.order_items",
    get_engine())
items_silver.shape

(112650, 7)

### order_payments

In [92]:
pay = pd.read_sql("SELECT * FROM bronze.olist_order_payments_dataset",get_engine())
pay.shape

(103886, 5)

In [100]:
pay.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [93]:
(pay == "").sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [94]:
# dublikatları yoxlayırıq
pay.duplicated().sum()

np.int64(0)

In [95]:
# Eyni sifariş və ödəniş sıra nömrəsinin təkrarını yoxlayırıq
pay.duplicated(subset=["order_id", "payment_sequential"]).sum()

np.int64(0)

In [96]:
pay[["payment_sequential","payment_installments","payment_value"]].dtypes

payment_sequential      object
payment_installments    object
payment_value           object
dtype: object

In [97]:
# Ödəniş sütunlarını uyğun rəqəm tiplərinə çeviririk
pay["payment_sequential"] = pd.to_numeric(pay["payment_sequential"], errors="coerce")
pay["payment_installments"] = pd.to_numeric(pay["payment_installments"], errors="coerce")
pay["payment_value"] = pd.to_numeric(pay["payment_value"], errors="coerce")

In [98]:
pay[["payment_sequential","payment_installments","payment_value"]].dtypes

payment_sequential        int64
payment_installments      int64
payment_value           float64
dtype: object

In [99]:
# Rəqəm sütunlarında boş dəyərləri yoxlayırıq
pay[["payment_sequential", "payment_installments", "payment_value"]].isna().sum()

payment_sequential      0
payment_installments    0
payment_value           0
dtype: int64

In [102]:
from sqlalchemy import types as sqltypes
pay = pd.read_sql("SELECT * FROM bronze.olist_order_payments_dataset",get_engine())

# Rəqəm sütunlarını uyğun tipə çeviririk
pay["payment_sequential"] = pd.to_numeric(pay["payment_sequential"])
pay["payment_installments"] = pd.to_numeric(pay["payment_installments"])
pay["payment_value"] = pd.to_numeric(pay["payment_value"])

# Məlumatları Silver cədvəlinə yazırıq
pay.to_sql("order_payments",get_engine(),schema="silver",if_exists="replace",index=False,
    dtype={
        "order_id": sqltypes.VARCHAR(32),
        "payment_sequential": sqltypes.SMALLINT(),
        "payment_type": sqltypes.VARCHAR(20),
        "payment_installments": sqltypes.SMALLINT(),
        "payment_value": sqltypes.DECIMAL(10, 2)})

393

In [103]:
pay_silver = pd.read_sql("SELECT * FROM silver.order_payments",get_engine())

pay_silver.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


### holidays

In [104]:
hol = pd.read_sql("SELECT * FROM bronze.api_holidays",get_engine())
hol.shape

(42, 9)

In [105]:
hol.head()

,date,localName,name,countryCode,fixed,global,counties,launchYear,types
0,2016-01-01,Confraternização Universal,New Year's Day,BR,False,True,None,None,"[""Public""]"
1,2016-02-08,Carnaval,Carnival,BR,False,True,None,None,"[""Bank"", ""Optional""]"
2,2016-02-09,Carnaval,Carnival,BR,False,True,None,None,"[""Bank"", ""Optional""]"
3,2016-03-25,Sexta-feira Santa,Good Friday,BR,False,True,None,None,"[""Public""]"
4,2016-03-27,Domingo de Páscoa,Easter Sunday,BR,False,True,None,None,"[""Public""]"


In [109]:
(hol == "").sum()

date           0
localName      0
name           0
countryCode    0
fixed          0
global         0
counties       0
launchYear     0
types          0
dtype: int64

In [114]:
# dublikatları yoxlayaq 
hol.duplicated().sum()

np.int64(0)

In [110]:
hol.isna().sum()

date            0
localName       0
name            0
countryCode     0
fixed           0
global          0
counties       39
launchYear     42
types           0
dtype: int64

In [112]:
# Bayram məlumatlarının sütun tiplərini yoxlayırıq
hol.dtypes

date           object
localName      object
name           object
countryCode    object
fixed            bool
global           bool
counties       object
launchYear     object
types          object
dtype: object

In [113]:
# Bayram tarixini tarix tipinə çeviririk
hol["date"] = pd.to_datetime(hol["date"], errors="coerce")

# Başlanğıc ilini rəqəm tipinə çeviririk
hol["launchYear"] = pd.to_numeric(hol["launchYear"], errors="coerce")

In [115]:
hol.dtypes

date           datetime64[ns]
localName              object
name                   object
countryCode            object
fixed                    bool
global                   bool
counties               object
launchYear            float64
types                  object
dtype: object

In [116]:
# Region və bayram kateqoriyası məlumatlarına baxırıq
hol[["counties", "types"]].head(10)

,counties,types
0,None,"[""Public""]"
1,None,"[""Bank"", ""Optional""]"
2,None,"[""Bank"", ""Optional""]"
3,None,"[""Public""]"
4,None,"[""Public""]"
5,None,"[""Public""]"
6,None,"[""Public""]"
7,None,"[""Public""]"
8,"[""BR-SP""]","[""Public""]"
9,None,"[""Public""]"


In [117]:
# Region və bayram növləri siyahılarını mətnə çeviririk
import json

hol["counties"] = hol["counties"].apply(
    lambda x: json.dumps(x) if x is not None else None)
hol["types"] = hol["types"].apply(json.dumps)

In [118]:
from sqlalchemy import types as sqltypes
import json

hol = pd.read_sql("SELECT * FROM bronze.api_holidays",get_engine())

# Silver üçün lazım olan sütunları seçirik
silver_hol = pd.DataFrame({"holiday_date": pd.to_datetime(hol["date"], errors="coerce").dt.date,
    "local_name": hol["localName"],
    "name": hol["name"],
    "is_national": hol["global"],
    "state_code": hol["counties"].str.extract(r"BR-([A-Z]{2})")[0],
    "types": hol["types"].apply(
        lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)})

# Məlumatları Silver cədvəlinə yazırıq
silver_hol.to_sql("holidays",get_engine(),schema="silver",if_exists="replace",index=False,
    dtype={
        "holiday_date": sqltypes.DATE(),
        "local_name": sqltypes.NVARCHAR(100),
        "name": sqltypes.VARCHAR(100),
        "is_national": sqltypes.Boolean(),
        "state_code": sqltypes.CHAR(2),
        "types": sqltypes.VARCHAR(200)})

42

In [120]:
hol_silver = pd.read_sql("SELECT * FROM silver.holidays",get_engine())
hol_silver.head()

,holiday_date,local_name,name,is_national,state_code,types
0,2016-01-01,Confraternização Universal,New Year's Day,True,None,"[""Public""]"
1,2016-02-08,Carnaval,Carnival,True,None,"[""Bank"", ""Optional""]"
2,2016-02-09,Carnaval,Carnival,True,None,"[""Bank"", ""Optional""]"
3,2016-03-25,Sexta-feira Santa,Good Friday,True,None,"[""Public""]"
4,2016-03-27,Domingo de Páscoa,Easter Sunday,True,None,"[""Public""]"
